# Практика 20. Лінійне програмування: постановка задачі, розв'язання засобами scipy

**Варіант 1: Київ**

Мета: сформулювати виробничу задачу лінійного програмування, знайти план максимального прибутку, визначити активні ресурсні обмеження та перевірити ефект їхньої зміни.

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import linprog

variant = 1
city = 'Київ'
profit_a, profit_b = 60.0, 45.0
resource_names = ['сировина, кг', 'робочий час, год', 'електроенергія, кВт·год']
resource_limits = np.array([300.0, 220.0, 95.0])
A_ub = np.array([
    [35.0, 30.0],
    [4.0, 2.0],
    [1.0, 1.0],
])
c = np.array([-profit_a, -profit_b])
print(f'Варіант {variant}: {city}')
print('Параметри задачі завантажено.')

Варіант 1: Київ
Параметри задачі завантажено.


## Завдання 1. Математична постановка власного варіанта

Нехай `x1` — кількість виробів А, `x2` — кількість виробів Б за тиждень.

**Максимізувати:** `P = 60*x1 + 45*x2` грн.

За умов:

- сировина: `35*x1 + 30*x2 <= 300` кг;
- робочий час: `4*x1 + 2*x2 <= 220` машино-год;
- електроенергія: `1*x1 + 1*x2 <= 95` кВт·год;
- `x1 >= 0`, `x2 >= 0`.

Для `linprog` три обмеження вже записані у формі `A_ub @ x <= b_ub`, а цільова функція буде інвертована для максимізації.

In [2]:
result = linprog(
    c,
    A_ub=A_ub,
    b_ub=resource_limits,
    bounds=[(0, None), (0, None)],
    method='highs',
)
print(f'x = {result.x}')
print(f'Значення функції linprog = {result.fun:.4f}')
print(f'Справжній прибуток = {-result.fun:.4f} грн')
print(f'status = {result.status}')
print(f'message = {result.message}')
print(f'Розв’язок успішний? {result.status == 0}')
print(f'slack = {result.slack}')

x = [8.57142857 0.        ]
Значення функції linprog = -514.2857
Справжній прибуток = 514.2857 грн
status = 0
message = Optimization terminated successfully. (HiGHS Status 7: Optimal)
Розв’язок успішний? True
slack = [  0.         185.71428571  86.42857143]


## Завдання 2. Розв'язання через `linprog`

`linprog` мінімізує цільову функцію, тому для максимізації прибутку використовуємо від'ємні коефіцієнти прибутку.

In [3]:
print(f'Виріб A: {result.x[0]:.4f} од./тиждень')
print(f'Виріб B: {result.x[1]:.4f} од./тиждень')
print(f'Максимальний прибуток: {-result.fun:.4f} грн/тиждень')
print('Для реального цілочислового виробництва потрібна окрема цілочислова оптимізація, а не механічне округлення.')

Виріб A: 8.5714 од./тиждень
Виріб B: 0.0000 од./тиждень
Максимальний прибуток: 514.2857 грн/тиждень
Для реального цілочислового виробництва потрібна окрема цілочислова оптимізація, а не механічне округлення.


## Завдання 3. Інтерпретація оптимального розв'язку

Перетворюємо результат `res.x` на кількість виробів А і Б на тиждень та `-res.fun` на максимальний прибуток. Дробовий розв'язок є коректним для безперервної ЛП; просте округлення не гарантує допустимості й оптимальності цілочислового плану.

In [4]:
usage = A_ub @ result.x
slack_table = pd.DataFrame({
    'ресурс': resource_names,
    'ліміт': resource_limits,
    'використано': usage,
    'slack': result.slack,
    'активний': np.isclose(result.slack, 0),
})
print(slack_table.round(4))
print('Активні ресурси:', ', '.join(slack_table.loc[slack_table['активний'], 'ресурс']))
print('Ресурси із запасом:', ', '.join(slack_table.loc[~slack_table['активний'], 'ресурс']))

                    ресурс  ліміт  використано     slack  активний
0             сировина, кг  300.0     300.0000    0.0000      True
1         робочий час, год  220.0      34.2857  185.7143     False
2  електроенергія, кВт·год   95.0       8.5714   86.4286     False
Активні ресурси: сировина, кг
Ресурси із запасом: робочий час, год, електроенергія, кВт·год


## Завдання 4. Активні обмеження

`res.slack` показує невикористаний запас кожного ресурсу у тій самій послідовності, що й обмеження. Slack, близький до нуля, означає активне обмеження та вузьке місце.

In [5]:
def solve_with_limits(limits):
    return linprog(
        c,
        A_ub=A_ub,
        b_ub=limits,
        bounds=[(0, None), (0, None)],
        method='highs',
    )

active_index = int(np.argmin(result.slack))
inactive_index = int(np.argmax(result.slack))
active_limits = resource_limits.copy()
active_limits[active_index] *= 1.15
inactive_limits = resource_limits.copy()
inactive_limits[inactive_index] *= 1.50

active_result = solve_with_limits(active_limits)
inactive_result = solve_with_limits(inactive_limits)

print(f'Активний ресурс для what-if: {resource_names[active_index]}')
print(f'Новий план: A = {active_result.x[0]:.4f}, B = {active_result.x[1]:.4f}, прибуток = {-active_result.fun:.4f} грн')
print(f'Базовий прибуток: {-result.fun:.4f} грн')
print(f'Зміна прибутку: {-active_result.fun + result.fun:.4f} грн')
print(f'Неактивний ресурс для what-if: {resource_names[inactive_index]}')
print(f'Новий план: A = {inactive_result.x[0]:.4f}, B = {inactive_result.x[1]:.4f}, прибуток = {-inactive_result.fun:.4f} грн')
print(f'Прибуток змінився після збільшення неактивного ресурсу? {not np.isclose(inactive_result.fun, result.fun)}')

Активний ресурс для what-if: сировина, кг
Новий план: A = 9.8571, B = 0.0000, прибуток = 591.4286 грн
Базовий прибуток: 514.2857 грн
Зміна прибутку: 77.1429 грн
Неактивний ресурс для what-if: робочий час, год
Новий план: A = 8.5714, B = 0.0000, прибуток = 514.2857 грн
Прибуток змінився після збільшення неактивного ресурсу? False


## Завдання 5. What-if: зміна ресурсного обмеження

Автоматично збільшуємо на 15% один активний ресурс і на 50% один неактивний ресурс. Порівнюємо новий план і прибуток із базовим розв'язком.

## Контрольні питання

**1. Чому `linprog` мінімізує?** Функція має стандартну постановку мінімізації. Для максимізації прибутку передаємо `c = [-profit_A, -profit_B]`, а після розв'язання відновлюємо прибуток як `-res.fun`.

**2. Навіщо множити `>=` на `-1`?** `linprog` приймає нерівності лише у формі `A_ub @ x <= b_ub`. Тому обмеження «не менше» треба еквівалентно переписати зі зміненими знаками.

**3. Активне й неактивне обмеження.** Активне має slack, близький до нуля, і є вузьким місцем: його розширення потенційно збільшує прибуток. Неактивне має запас і саме по собі не обмежує поточний оптимум.

**4. Що очікуємо у what-if?** Збільшення активного ресурсу може змінити план і збільшити прибуток. Збільшення неактивного ресурсу зазвичай не змінює оптимум, бо вузьке місце залишається іншим.

## Підсумок

- Математичну постановку задачі записано до програмного розв'язання.
- Максимізацію прибутку реалізовано через мінімізацію `-прибуток` у `linprog`.
- Оптимальний план і активні обмеження інтерпретовано в одиницях виробництва та ресурсів.
- Перевірено what-if для активного й неактивного ресурсів.